# 1. Load dataset 

In [1]:
import pandas as pd
df = pd.read_csv('merged_data_1000.csv')
df.head()

,review,sentiment
0,"For Daniel Auteuil, `Queen Margot' was much be...",negative
1,Spoilers abound. You have been warned.<br /><b...,negative
2,"Where do I start? The plot of the movie, which...",negative
3,There was a genie played by Shaq His name was ...,negative
4,Its a very good comedy movie.Ijust liked it.I ...,positive


# 2. Preprocessing

In [3]:
import contractions
from bs4 import BeautifulSoup
from nltk . stem import WordNetLemmatizer
from nltk . corpus import stopwords
import re
import string
import nltk
nltk . download('stopwords')
nltk . download('wordnet')

[nltk_data] Downloading package stopwords to C:\Users\Dao Thi
[nltk_data]     Huyen\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\Dao Thi
[nltk_data]     Huyen\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [4]:
duplicated_df = df[df.duplicated()]
print(duplicated_df)

Empty DataFrame
Columns: [review, sentiment]
Index: []


In [5]:
df = df.drop_duplicates()

In [6]:
df.describe()

,review,sentiment
count,1000,1000
unique,1000,2
top,"For Daniel Auteuil, `Queen Margot' was much be...",negative
freq,1,500


## Data Cleaning

In [7]:
stop = set(stopwords.words('english'))
print(stop)

{'other', 'her', 'until', "she'll", 'couldn', 'i', "you'll", 'herself', "don't", 'what', 'more', 'hadn', 'theirs', 'between', 'this', 'why', 'your', 'being', 's', "that'll", 'who', 'ma', 'you', "we're", 'too', "you'd", 'been', "she's", "they've", "hasn't", "hadn't", 'did', 'does', 'have', 'most', 'doing', "we've", 'during', 'and', "he's", 'him', 'myself', "it'd", 'haven', 'o', 'yourself', 'their', "she'd", 'my', 'is', 'again', "it's", 'the', 'there', "wasn't", 'were', 'weren', 'to', 'ours', 'about', 'himself', "we'll", 'both', 'shan', 'while', 'above', "won't", "you've", 'so', 'all', 'with', 'because', 'that', 'own', 'be', 'below', 'then', "you're", "it'll", "i'll", "i'm", 'are', 'm', "couldn't", 'hers', 'of', 'in', 'shouldn', "weren't", "mightn't", "didn't", 'but', 'themselves', 'wouldn', 'can', 'd', "i've", 'which', 'than', 'into', 'under', 'y', 'if', 'nor', 'wasn', 'was', "should've", 'here', 't', "shouldn't", 'by', 'our', 'up', "needn't", "i'd", 'at', "they're", 'each', 'having', '

In [8]:
# "I'm" → "I am"
def expand_contractions(text):
    return contractions.fix(text)

preprocess_text xử lý:

- Xóa HTML, emoji, URL

- Xử lý dấu câu

- Thu nhỏ chữ (lowercase)

- Lemmatize (đưa từ về dạng gốc)

- Xóa stopwords (các từ dư thừa như "the", "is", "and"...)

In [9]:
def preprocess_text(text):
    wl = WordNetLemmatizer() # Chuẩn hóa từ về dạng gốc, ví dụ "running" → "run".
    soup = BeautifulSoup(text, "html.parser")  # "<div> I can't <b>swim</b>!</div>"
    text = soup.get_text()                     # "I can't swim!"
    text = expand_contractions(text)           # "I can not swim!" 
    emoji_clean = re.compile("["
                             u"\U0001F600-\U0001F64F"  # Các emoji cảm xúc 
                             u"\U0001F300-\U0001F5FF"  # Biểu tượng tự nhiên, vật dụng 
                             u"\U0001F680-\U0001F6FF"  # Xe cộ, giao thông 
                             u"\U0001F1E0-\U0001F1FF"  # Quốc kỳ các nước
                             u"\U00002702-\U000027B0"  # Các ký hiệu nhỏ
                             u"\U000024C2-\U0001F251"  # Các ký hiệu hỗn hợp khác
                             "]+", flags=re. UNICODE)
    text = emoji_clean.sub(r'', text)                   # emoji => '' 
    text = re.sub(r'\.(?=\S)', '. ', text)   # "This is good.This is bad" → "This is good. This is bad"
    text = re.sub(r'http\S+', '', text)  # remove urls
    text = "". join([word.lower()
                     for word in text if word not in string.punctuation])  # Bỏ hết dấu câu (string.punctuation) như . , ; ! ?...
                                                                           # Đồng thời chuyển text thành chữ thường (lowercase).
                                                                           # "this is good this is bad"
    text = " ". join([wl.lemmatize(word) for word in text.split()
                     if word not in stop and word.isalpha()])              # Loại bỏ từ stop và số
                                                                           # chuẩn hóa từ về dạng gốc
    return text

In [10]:
df['review'][0]

"For Daniel Auteuil, `Queen Margot' was much better. For Nastassja Kinski, `Paris, Texas' was much better. The biggest disappointments were from Chris Menges (`CrissCross' and `A World Apart' cannot even be compared with this one), and Goran Bregovic for use of a version of the same musical theme from `Queen Margot' for this movie (Attention to the end of the film). If this was an American pop movie, I would not feel surprised at all; but for a European film with more independent actors and director, a similar common approach about child abuse with no original insight is very simple-minded and disappointing. There are those bad guys who kidnap and sell the underage people. There are those poor children who hate people selling them and wait to be saved by someone. And finally, there is that big hero who kills all the bad guys and saves these poor children from bad guys. Every character is shown in simple black and white terms: the good versus the evil. Plus, from the very beginning, I c

In [11]:
preprocess_text(df['review'][0])

'daniel auteuil queen margot much better nastassja kinski paris texas much better biggest disappointment chris menges crisscross world apart cannot even compared one goran bregovic use version musical theme queen margot movie attention end film american pop movie would feel surprised european film independent actor director similar common approach child abuse original insight simpleminded disappointing bad guy kidnap sell underage people poor child hate people selling wait saved someone finally big hero kill bad guy save poor child bad guy every character shown simple black white term good versus evil plus beginning could understand story would end end history child sexual abuse believe difficult issue child molestation paedophilia much complex portrayed original movie think movie disturbing disappointing'

In [12]:
df['review'] = df['review'].apply(preprocess_text)

In [13]:
df['review']

0      daniel auteuil queen margot much better nastas...
1      spoiler abound warned thoroughly disappointed ...
2      start plot movie love two high school student ...
3      genie played shaq name kazaam whack rhyme corn...
4      good comedy movie ijust liked know love movie ...
                             ...                        
995    quiet sweet beutifully nostalgic movie confron...
996    first believe movie much appreciated viewer ac...
997    reef play haji murad hero russia badly dubbed ...
998    williams family live ranch located middle remo...
999    like first movie iffy dialogue weaker acting s...
Name: review, Length: 1000, dtype: object

In [25]:
word_count_df = pd.DataFrame({
    'word': vocabs,
    'word_count': word_counts
})

In [26]:
word_count_df

,word,word_count
0,movie,20194
1,film,18551
2,one,10955
3,like,7925
4,would,6285
...,...,...
59493,pomme,1
59494,modestjust,1
59495,cérémonie,1
59496,dokken,1


# Text Encoding

In [14]:
from sklearn . model_selection import train_test_split
from sklearn . feature_extraction . text import TfidfVectorizer
from sklearn . preprocessing import LabelEncoder

label_encode = LabelEncoder()
x_data = df['review']
y_data = label_encode.fit_transform(df['sentiment'])
x_train, x_test, y_train, y_test = train_test_split(
    x_data, y_data, test_size=0.2, random_state=42)

In [15]:
tfidf_vectorizer = TfidfVectorizer(max_features=10)  # lấy 150 từ phổ biến nhất làm từ điển
tfidf_vectorizer.fit(x_train, y_train)

x_train_encoded = tfidf_vectorizer.transform(x_train)
x_test_encoded = tfidf_vectorizer.transform(x_test)

In [16]:
x_train_encoded.shape

(800, 10)

In [17]:
x_train_encoded[0]

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 5 stored elements and shape (1, 10)>

In [18]:
import pandas as pd

# Chọn các index câu cần in
indices = [11, 12, 13, 14, 15]  # Bạn thay đổi tùy ý

# Lấy từ vựng
feature_names = tfidf_vectorizer.get_feature_names_out()

# Khởi tạo DataFrame rỗng
tfidf_table = pd.DataFrame(columns=["Câu Gốc", "Label"] + list(feature_names))

# Duyệt qua từng câu
for idx in indices:
    tfidf_vector = x_train_encoded[idx].toarray().flatten()
    sentence = x_train.iloc[idx]
    label = y_train[idx]  # <<-- Thêm dòng này để lấy nhãn
    
    # Tạo 1 dòng mới: Câu + Label + các tf-idf
    row = {"Câu Gốc": sentence, "Label": label}
    for i, word in enumerate(feature_names):
        row[word] = tfidf_vector[i]
    
    tfidf_table = pd.concat([tfidf_table, pd.DataFrame([row])], ignore_index=True)

# Lưu vào file CSV
tfidf_table.to_csv("tfidf_output_with_label.csv", index=False, encoding='utf-8-sig')

print("Đã lưu file tfidf_output_with_label.csv thành công!")


Đã lưu file tfidf_output_with_label.csv thành công!


C:\Users\Dao Thi Huyen\AppData\Local\Temp\ipykernel_4816\3139643233.py:23: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  tfidf_table = pd.concat([tfidf_table, pd.DataFrame([row])], ignore_index=True)


In [19]:
import pandas as pd
df_2 = pd.read_csv('tfidf_output_with_label.csv')
df_2.head()

,Câu Gốc,Label,character,even,film,good,like,make,movie,one,time,would
0,freddys dead final nightmare last film feature...,1,0.000000,0.000000,0.302841,0.399365,0.000000,0.412690,0.279581,0.601523,0.372138,0.000000
1,movie took surprise opening credit sequence fe...,1,0.515924,0.000000,0.391229,0.000000,0.597942,0.000000,0.361181,0.259028,0.160251,0.000000
2,ever put review bad taste quite funny genius f...,1,0.000000,0.000000,0.000000,0.438381,0.381054,0.453009,0.000000,0.330145,0.408495,0.426074
3,show happened screen saver got hand taking use...,0,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,kevin kline offer brilliant comic turn comedy ...,1,0.212140,0.436137,0.643468,0.000000,0.368796,0.219218,0.148512,0.319524,0.000000,0.206184


In [20]:
# Bỏ cột 'Câu Gốc' ra vì nó không phải feature
feature_df = df_2.drop(columns=["Câu Gốc", "Label"])

# Tạo dict lưu kết quả
unique_values = {}

# Duyệt từng feature
for col in feature_df.columns:
    unique_vals = feature_df[col].unique()
    unique_values[col] = sorted(unique_vals)

# Chuyển dict thành DataFrame cho dễ nhìn
unique_values_df = pd.DataFrame(dict([(k, pd.Series(v)) for k, v in unique_values.items()]))

# In kết quả
print(unique_values_df)

# Nếu muốn lưu ra file CSV
unique_values_df.to_csv("unique_tf_idf_values.csv", index=False, encoding='utf-8-sig')
print("Đã lưu file unique_tf_idf_values.csv!")


   character      even      film      good      like      make     movie  \
0   0.000000  0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   
1   0.212140  0.436137  0.302841  0.399365  0.368796  0.219218  0.148512   
2   0.515924  1.000000  0.391229  0.438381  0.381054  0.412690  0.279581   
3        NaN       NaN  0.643468       NaN  0.597942  0.453009  0.361181   
4        NaN       NaN       NaN       NaN       NaN       NaN       NaN   

        one      time     would  
0  0.000000  0.000000  0.000000  
1  0.259028  0.160251  0.206184  
2  0.319524  0.372138  0.426074  
3  0.330145  0.408495       NaN  
4  0.601523       NaN       NaN  
Đã lưu file unique_tf_idf_values.csv!


In [21]:
import pandas as pd
df_3 = pd.read_csv('unique_tf_idf_values.csv')
df_3.head()

,character,even,film,good,like,make,movie,one,time,would
0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0.212140,0.436137,0.302841,0.399365,0.368796,0.219218,0.148512,0.259028,0.160251,0.206184
2,0.515924,1.000000,0.391229,0.438381,0.381054,0.412690,0.279581,0.319524,0.372138,0.426074
3,NaN,NaN,0.643468,NaN,0.597942,0.453009,0.361181,0.330145,0.408495,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.601523,NaN,NaN


In [24]:
import pandas as pd

# Bỏ cột 'Câu Gốc'
feature_df = df_2.drop(columns=["Câu Gốc", "Label"])

# Dict lưu kết quả
mean_values = {}

# Duyệt từng feature
for col in feature_df.columns:
    unique_vals = sorted(feature_df[col].unique())
    mean_list = [(unique_vals[i] + unique_vals[i+1]) / 2 for i in range(len(unique_vals)-1)]
    mean_values[col] = mean_list

# Chuyển dict thành DataFrame
mean_values_df = pd.DataFrame(dict([(k, pd.Series(v)) for k, v in mean_values.items()]))

# In kết quả
print(mean_values_df)

# Lưu ra file CSV
mean_values_df.to_csv("mean_of_unique_pairs.csv", index=False, encoding='utf-8-sig')
print("Đã lưu file mean_of_unique_pairs.csv!")


   character      even      film      good      like      make     movie  \
0   0.106070  0.218068  0.151421  0.199682  0.184398  0.109609  0.074256   
1   0.364032  0.718068  0.347035  0.418873  0.374925  0.315954  0.214046   
2        NaN       NaN  0.517349       NaN  0.489498  0.432849  0.320381   
3        NaN       NaN       NaN       NaN       NaN       NaN       NaN   

        one      time     would  
0  0.129514  0.080125  0.103092  
1  0.289276  0.266195  0.316129  
2  0.324835  0.390317       NaN  
3  0.465834       NaN       NaN  
Đã lưu file mean_of_unique_pairs.csv!


In [25]:
import pandas as pd
df_4 = pd.read_csv('mean_of_unique_pairs.csv')
df_4.head()

,character,even,film,good,like,make,movie,one,time,would
0,0.106070,0.218068,0.151421,0.199682,0.184398,0.109609,0.074256,0.129514,0.080125,0.103092
1,0.364032,0.718068,0.347035,0.418873,0.374925,0.315954,0.214046,0.289276,0.266195,0.316129
2,NaN,NaN,0.517349,NaN,0.489498,0.432849,0.320381,0.324835,0.390317,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.465834,NaN,NaN
